In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [26]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [14]:
train_df.head()

,SampleID,departure_time,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class,delay_minutes
0,4227,02:58,788.12,103.94,2,sunny,Fri,0,3,61.463731,intermediate,11
1,4676,05:19,408.42,96.60,7,sunny,Sat,0,13,125.058439,premium,17
2,800,18:44,440.24,92.54,1,sunny,Sun,0,12,178.797255,standard,0
3,3671,06:27,345.01,104.57,5,sunny,Sat,0,11,137.304807,standard,5
4,4193,22:09,729.77,82.12,7,sunny,Wed,0,6,193.314124,premium,27


In [50]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SampleID        4000 non-null   int64  
 1   hour            4000 non-null   int32  
 2   minute          4000 non-null   int32  
 3   distance_km     4000 non-null   float64
 4   avg_speed_kmh   4000 non-null   float64
 5   num_stops       4000 non-null   int64  
 6   weather         4000 non-null   object 
 7   weekday         4000 non-null   object 
 8   special_events  4000 non-null   int64  
 9   num_cars        4000 non-null   int64  
 10  ticket_price    4000 non-null   float64
 11  comfort_class   4000 non-null   object 
 12  delay_minutes   4000 non-null   int64  
dtypes: float64(3), int32(2), int64(5), object(3)
memory usage: 375.1+ KB


In [49]:
"""Column re-formatting and data type changing for ML"""

train_df['departure_time'] = pd.to_datetime(train_df['departure_time'], format='%H:%M')
test_df['departure_time'] = pd.to_datetime(test_df['departure_time'], format='%H:%M')

train_df['hour'] = train_df['departure_time'].dt.hour
train_df['minute'] = train_df['departure_time'].dt.minute

test_df['hour'] = test_df['departure_time'].dt.hour
test_df['minute'] = test_df['departure_time'].dt.minute

train_df = train_df.drop(columns=['departure_time'])
test_df = test_df.drop(columns=['departure_time'])

t_minute = train_df.pop('minute')
train_df.insert(1, 'minute', t_minute)

t_hour = train_df.pop('hour')
train_df.insert(1, 'hour', t_hour)

test_minute = test_df.pop('minute')
test_df.insert(1, 'minute', test_minute)

test_hour = test_df.pop('hour')
test_df.insert(1, 'hour', test_hour)

In [51]:
test_df.head()

,SampleID,hour,minute,distance_km,avg_speed_kmh,num_stops,weather,weekday,special_events,num_cars,ticket_price,comfort_class
0,1501,15,44,470.26,91.85,9,sunny,Mon,1,6,190.082409,standard
1,2586,4,58,237.40,94.70,0,snow,Tue,0,5,138.302023,standard
2,2653,14,50,95.05,117.78,4,sunny,Thu,0,7,127.085555,intermediate
3,1055,13,22,480.48,55.15,1,sunny,Tue,0,12,90.206777,standard
4,705,8,39,303.02,55.40,1,sunny,Sat,0,6,109.905027,standard


In [52]:
"""Regression task - Used Advanced LGBMRegressor model with internal lgb.Dataset class constructing for the lgb.train function"""

import lightgbm as lgb
from lightgbm import LGBMRegressor

X_train = train_df.drop(columns=['delay_minutes', 'SampleID'])
y_train = train_df['delay_minutes']

X_test = test_df.drop(columns=['SampleID'])

train_cat = train_df.select_dtypes(include=['object']).columns
test_cat = test_df.select_dtypes(include=['object']).columns

for col in train_cat:
  X_train[col] = X_train[col].astype('category')

for col in test_cat:
  X_test[col] = X_test[col].astype('category')

train_data = lgb.Dataset(data=X_train, label=y_train, categorical_feature=list(train_cat))
test_data = lgb.Dataset(data=X_test, categorical_feature=list(test_cat))

model = LGBMRegressor(objective='regression', n_jobs=-1, random_state=42)

print(model.get_params())

bst = lgb.train(params=model.get_params(), train_set=train_data)

{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'num_leaves': 31, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0}
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008700 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 891
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 11
[LightGBM] [Info] Start training from score 16.627500


In [61]:
y_preds = bst.predict(X_test)

np.set_printoptions(precision=2)

y_preds[:50]

array([27.08, 10.6 ,  1.2 , 13.68,  5.07, 10.44, 31.05, 18.74, 23.68,
        6.81,  9.36,  8.69, 21.66, 16.24, 21.09, 26.84, 12.58, 10.94,
        2.88,  6.13, 32.88, 20.81, 26.76, 18.59, 33.16, 35.41, 25.62,
       23.04, 10.28, 33.77, -0.54, 13.29, 26.86,  5.89, 11.34, 23.14,
       19.77, 38.96,  8.04, 10.96, 20.87, 22.87, 24.14, 11.75,  7.63,
       25.66, 18.69,  7.02, 10.5 , -1.01])

In [69]:
"""Curious to see the spread differences between the two"""

print(f'train_df std: {round(train_df['delay_minutes'].std(), 3)} vs predicted std: {round(y_preds.std(), 3)}')

train_df std: 10.739 vs predicted std: 9.448


In [66]:
"""Submission DataFrame constructing + downloading"""

submission = pd.DataFrame({
    'SampleID': test_df['SampleID'],
    'delay_minutes': y_preds.astype(int),
})

submission.head(3)

,SampleID,delay_minutes
0,1501,27
1,2586,10
2,2653,1


In [67]:
submission.to_csv('submission.csv', index=False)
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>